# Full Hand-Sequence Gesture Classification (LSTM)

This notebook trains a gesture classifier using **full-hand landmark sequences** (21 keypoints × 2D per frame),
with **TIME_STEPS = 16** frames and **NUM_CLASSES = 3**.

## Expected dataset
- CSV file: `model/full_sequence_classifier/full_sequence.csv`
- Row format: `[class_id, frame1(42 vals), frame2(42 vals), ..., frame16(42 vals)]` => total 1 + 16×42 columns.
- Labels: `model/full_sequence_classifier/full_sequence_classifier_label.csv` (one label per line, row index = class id).

> Make sure your data were collected with the same preprocessing as runtime (relative to wrist + normalization).


In [ ]:
import os, time, csv
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

# Paths and hyperparams
DATASET = 'model/full_sequence_classifier/full_sequence.csv'
LABELS_PATH = 'model/full_sequence_classifier/full_sequence_classifier_label.csv'
MODEL_DIR = 'model/full_sequence_classifier'
MODEL_SAVE_PATH = os.path.join(MODEL_DIR, 'full_sequence_classifier.hdf5')
TFLITE_PATH = os.path.join(MODEL_DIR, 'full_sequence_classifier.tflite')

NUM_CLASSES = 3
TIME_STEPS = 16
DIM_PER_FRAME = 42   # 21 keypoints × (x,y)
INPUT_LEN = TIME_STEPS * DIM_PER_FRAME
RANDOM_SEED = 42
BATCH_SIZE = 64
EPOCHS = 200

os.makedirs(MODEL_DIR, exist_ok=True)
print(tf.__version__)


In [ ]:
# Load dataset
usecols = list(range(1, INPUT_LEN + 1))
X = np.loadtxt(DATASET, delimiter=',', dtype='float32', usecols=usecols)
y = np.loadtxt(DATASET, delimiter=',', dtype='int32', usecols=(0,))
print('X shape:', X.shape, 'y shape:', y.shape)

# Sanity checks
assert X.shape[1] == INPUT_LEN, (X.shape, INPUT_LEN)
assert y.ndim == 1

X_train, X_test, y_train, y_test = train_test_split(
    X, y, train_size=0.8, random_state=RANDOM_SEED, stratify=y
)
X_train.shape, X_test.shape


In [ ]:
# Build model (LSTM)
use_lstm = True
if use_lstm:
    model = tf.keras.Sequential([
        tf.keras.layers.InputLayer(input_shape=(INPUT_LEN,)),
        tf.keras.layers.Reshape((TIME_STEPS, DIM_PER_FRAME)),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.LSTM(64),
        tf.keras.layers.Dropout(0.5),
        tf.keras.layers.Dense(64, activation='relu'),
        tf.keras.layers.Dense(NUM_CLASSES, activation='softmax'),
    ])
else:
    model = tf.keras.Sequential([
        tf.keras.layers.InputLayer(input_shape=(INPUT_LEN,)),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Dense(128, activation='relu'),
        tf.keras.layers.Dropout(0.5),
        tf.keras.layers.Dense(64, activation='relu'),
        tf.keras.layers.Dense(NUM_CLASSES, activation='softmax'),
    ])
model.summary()

model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

callbacks = [
    tf.keras.callbacks.ModelCheckpoint(MODEL_SAVE_PATH, save_best_only=True),
    tf.keras.callbacks.EarlyStopping(patience=20, restore_best_weights=True),
]

hist = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    verbose=1,
    callbacks=callbacks,
)


In [ ]:
# Plot curves
plt.figure(figsize=(10,4))
plt.subplot(1,2,1); plt.plot(hist.history['loss'], label='train'); plt.plot(hist.history['val_loss'], label='val'); plt.title('Loss'); plt.legend();
plt.subplot(1,2,2); plt.plot(hist.history['accuracy'], label='train'); plt.plot(hist.history['val_accuracy'], label='val'); plt.title('Accuracy'); plt.legend();
plt.show()

test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print('Test acc:', test_acc)


In [ ]:
# (Optional) Confusion matrix
try:
    from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
    y_pred = model.predict(X_test, verbose=0).argmax(axis=1)
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(cm)
    disp.plot(xticks_rotation=45)
except Exception as e:
    print('Confusion matrix skipped:', e)


In [ ]:
# Export TFLite
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()
with open(TFLITE_PATH, 'wb') as f:
    f.write(tflite_model)
print('Saved TFLite to', TFLITE_PATH)


In [ ]:
# Quick sanity check using TFLite interpreter
# (The actual runtime module also performs this.)
import numpy as np
try:
    from tensorflow.lite.python.interpreter import Interpreter
except Exception:
    import tflite_runtime.interpreter as tflite
    Interpreter = tflite.Interpreter

interpreter = Interpreter(model_path=TFLITE_PATH)
interpreter.allocate_tensors()
input_index = interpreter.get_input_details()[0]['index']
output_index = interpreter.get_output_details()[0]['index']

x0 = X_test[:1].astype('float32')
interpreter.set_tensor(input_index, x0)
interpreter.invoke()
y0 = interpreter.get_tensor(output_index)[0]
print('TFLite output:', y0, 'argmax=', int(np.argmax(y0)))
